In [2]:
#Copyright 2026 Mücahit Sahin
#
#Licensed under the Apache License, Version 2.0 (the "License");
#you may not use this file except in compliance with the License.
#You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
#Unless required by applicable law or agreed to in writing, software
#distributed under the License is distributed on an "AS IS" BASIS,
#WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#See the License for the specific language governing permissions and
#limitations under the License.

import os
import sys
import shutil
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from autogluon.tabular import TabularPredictor

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from utils import hf_model
from utils import gpt
from utils import splitting
from utils import metrics
from utils import feature_generator
from utils import feature_extraction


import warnings
warnings.filterwarnings("ignore")

In [2]:
class_map = {
    0: 'numeric',
    1: 'categorical',
    2: 'datetime',
    3: 'sentence',
    4: 'url',
    5: 'embedded-number',
    6: 'list',
    7: 'not-generalizable',
    8: 'context-specific'
}

In [4]:
df_all = pd.read_csv('../data/Benchmark-Labeled-Data/all_data.csv')
metadata = pd.read_csv('../data/Benchmark-Labeled-Data/Metadata/meta_data.csv')

In [5]:
df_merged = pd.merge(df_all, metadata, on='Record_id', how='left', sort=False)

In [6]:
def check_if_exists(df_all):
    previous_file = ""
    file = ""
    path = '../data/Benchmark-Labeled-Data/Rawdata/'
    labels = []
    for index, row in df_merged.iterrows():
        previous_file = file
        file = path + str(row['Record_id']) + '/' + row['name']
        
        if os.path.isfile(file):
            labels.append(df_all.loc[index, 'y_act'])
        else:
            labels.append('Dataset does not exist')
            
    df_all['exist_or_not'] = labels
    return df_all

In [7]:
df_all = check_if_exists(df_all)

In [8]:
df_all['exist_or_not'].value_counts()

Dataset does not exist    9921
Name: exist_or_not, dtype: int64

In [9]:
df_all['y_act'].value_counts()

numeric              3612
categorical          2310
not-generalizable    1088
context-specific      889
datetime              678
embedded-number       568
sentence              382
list                  247
url                   147
Name: y_act, dtype: int64

In [10]:
featurized_data = pd.read_csv('../results/featurized_data_with_llm_features.csv')

In [10]:
featurized_data.insert(0, 'Attribute_name', df_all['Attribute_name'])
featurized_data.insert(1, 'y_act', df_all['y_act'])

In [11]:
gpt_5_4_fewshot_features = [f for f in list(featurized_data.columns) if "gpt_5_4" in str(f).lower() and "fewshot" in str(f).lower() and not "all_at_once" in str(f).lower()]

sortinghat_features = list(featurized_data.columns[2:1478])

base_features_with_regex = sortinghat_features[:25]
base_features = [s for s in sortinghat_features[:25] if not 'is_' in s and not 'has_' in s]

In [12]:
featurized_data = featurized_data.drop(df_all.loc[df_all['exist_or_not']=='Dataset does not exist'].index).reset_index(drop=True)
df_all = df_all.drop(df_all.loc[df_all['exist_or_not']=='Dataset does not exist'].index).reset_index(drop=True)

# Performance across 100 resample splits

In [13]:
def resample_and_evaluate(n_splits, feature_sets, featurized_data, raw_data):
    # Create a list to store the results for each feature set on each test set
    results = []
    results_df = pd.DataFrame()
    save_path = ""
    model_names = {
        'RF': 'RandomForest',
        'XGB': 'XGBoost',
        'CAT': 'CatBoost',
        'GBM': 'LightGBM',
        'NN_TORCH': 'NeuralNetTorch',
        'FASTAI': 'NeuralNetFastAI',
        'XT': 'ExtraTrees',
        'KNN': 'KNeighbors',
        'LR': 'LinearModel',
    }

    # limit to 100 resamples at max
    if n_splits > 100:
        n_splits = 100
    
    with open('../seeds.txt', 'r') as fp: # change the tolerance here
        seeds = fp.read().splitlines()
        seeds = [int(i) for i in seeds] # convert back to int
        fp.close()

    feature_set = list(feature_sets.keys())[0]
    features_to_learn = feature_sets[feature_set]['features']
    feature_set_data = featurized_data.copy()
    llms = ('gpt_5_4', 'gemma_4', 'qwen_3_5')
    llm_cols = [fl for fl in features_to_learn if fl.lower().startswith(llms)]
    boolean_cols = ['has_delimiters', 'has_url', 'has_email', 'has_date', 'is_list', 'is_long_sentence']
        
    sentenceTF_vc = feature_extraction.SentenceTFVectorizer()
    feature_set_data = sentenceTF_vc.transform(featurized_data)
    features_to_learn = features_to_learn + ['SentenceTF_embedding_dim_'+ str(i) for i in range(384)]
    
    # create the number of resample splits
    for i in range(n_splits):
        print(f"\tResample {i+1} ... ", end="")

        seed = seeds[i]

        best_inner_model_save_path = 'results/Models/' + str(seed) + '/Inner_Loop/' + \
                                         feature_set.replace('+', '')
        best_inner_model_save_path = ' '.join(best_inner_model_save_path.split()).replace(' ', '_')
        
        best_outer_model_save_path = 'results/Models/' + str(seed) + '/Outer_Loop/' + \
                                     feature_set.replace('+', '')
        best_outer_model_save_path = ' '.join(best_outer_model_save_path.split()).replace(' ', '_')

        
        metric = {
            'Seed': seed,
            'F1-Score (Dev)': 0,
            'Precision (Dev)': 0,
            'Recall (Dev)': 0,
            'Accuracy (Dev)': 0,
            'F1-Score (Test)': 0,
            'Precision (Test)': 0,
            'Recall (Test)': 0,
            'Accuracy (Test)': 0,
            'class_numeric_F1-Score': 0,
            'class_numeric_Precision': 0,
            'class_numeric_Recall': 0,
            'class_numeric_Accuracy': 0,
            'class_datetime_F1-Score': 0,
            'class_datetime_Precision': 0,
            'class_datetime_Recall': 0,
            'class_datetime_Accuracy': 0,
            'class_other_F1-Score': 0,
            'class_other_Precision': 0,
            'class_other_Recall': 0,
            'class_other_Accuracy': 0
        }
        best_model = {
            'Name': None,
            'Parameters': None,
            'F1-Score (Dev)': 0,
            'Precision (Dev)': 0,
            'Recall (Dev)': 0,
            'Accuracy (Dev)': 0,
            'F1-Score (Test)': 0,
            'Precision (Test)': 0,
            'Recall (Test)': 0,
            'Accuracy (Test)': 0
        }

        # Models
        model = TabularPredictor(label="y_act",
                                 problem_type='multiclass',
                                 eval_metric='f1_macro',
                                 path=best_inner_model_save_path,
                                 verbosity=0)

        outer_fold = splitting.create_split(raw_data, test_size=0.2, target_col='y_act', group_col='Record_id', random_state=seed)
        train = raw_data.iloc[outer_fold["train"]].reset_index(drop=True)
        train_featurized = feature_set_data.iloc[outer_fold["train"]].reset_index(drop=True)
        test_featurized = feature_set_data.iloc[outer_fold["test"]].reset_index(drop=True)
        
        inner_fold = splitting.create_split(train, test_size=0.25, target_col='y_act', group_col='Record_id', random_state=seed)
        inner_train_featurized = train_featurized.iloc[inner_fold["train"]].reset_index(drop=True)
        dev_featurized = train_featurized.iloc[inner_fold["test"]].reset_index(drop=True)

        y_train = inner_train_featurized["y_act"]
        y_dev = dev_featurized["y_act"]

    
        # Training
        X_train = inner_train_featurized.filter(features_to_learn)
        # some columns are int, make all strings to escape error from model.fit
        X_train.columns = X_train.columns.astype(str)
        train_data = X_train.copy()
        train_data["y_act"] = y_train
        
        # Predicting
        X_dev = dev_featurized.filter(features_to_learn)
        # some columns are int, make all strings to escape error from model.fit
        X_dev.columns = X_dev.columns.astype(str)
        dev_data = X_dev.copy()
        dev_data["y_act"] = y_dev

        # Model
        model.fit(
            train_data=train_data,
            tuning_data=dev_data,
            use_bag_holdout=True,    # makes sure to use dev_data
            feature_generator=None,  # disables preprocessing
            presets='best_quality',
            hyperparameters={
                feature_sets[feature_set]['model']: {}
            },
            time_limit=600,
            num_stack_levels=0,
            num_bag_folds=0
        )
        
        y_pred = model.predict(dev_data)

        # Calculate performance of the model
        acc_dev = accuracy_score(y_dev, y_pred)
        f1_dev = f1_score(y_dev, y_pred, average='macro')
        prec_dev = precision_score(y_dev, y_pred, average='macro')
        rec_dev = recall_score(y_dev, y_pred, average='macro')

        best_model["Name"] = model_names.get(feature_sets[feature_set]['model'])
        best_model["Parameters"] = model._trainer.load_model(model.leaderboard(silent=True).iloc[0]['model']).params
        best_model["F1-Score (Dev)"] = f1_dev
        best_model["Precision (Dev)"] = prec_dev
        best_model["Recall (Dev)"] = rec_dev
        best_model["Accuracy (Dev)"] = acc_dev


        metric["F1-Score (Dev)"] = f1_dev
        metric["Precision (Dev)"] = prec_dev
        metric["Recall (Dev)"] = rec_dev
        metric["Accuracy (Dev)"] = acc_dev

        # Best Tuned Models
        tuned_model = TabularPredictor(label="y_act",
                                 problem_type='multiclass',
                                 eval_metric='f1_macro',
                                 path=best_outer_model_save_path,
                                 verbosity=0)

        y_train = train_featurized["y_act"]
        y_test = test_featurized["y_act"]

        # Training
        X_train = train_featurized.filter(features_to_learn)
        # some columns are int, make all strings to escape error from model.fit
        X_train.columns = X_train.columns.astype(str)
        train_data = X_train.copy()
        train_data["y_act"] = y_train
        
        # Predicting
        X_test = test_featurized.filter(features_to_learn)
        # some columns are int, make all strings to escape error from model.fit
        X_test.columns = X_test.columns.astype(str)
        test_data = X_test.copy()
        test_data["y_act"] = y_test


        # Model
        tuned_model.fit(
            train_data=train_data,
            feature_generator=None,  # disables preprocessing
            hyperparameters={feature_sets[feature_set]['model']: best_model["Parameters"]},
            excluded_model_types=[],  # ensures no other models are trained
            presets='best_quality',
            time_limit=600,
            num_stack_levels=0,
            num_bag_folds=0
        )

        y_pred = tuned_model.predict(test_data)

        # Calculate performance from best model
        acc_test = accuracy_score(y_test, y_pred)
        f1_test = f1_score(y_test, y_pred, average='macro')
        prec_test = precision_score(y_test, y_pred, average='macro')
        rec_test = recall_score(y_test, y_pred, average='macro')

        best_model["F1-Score (Test)"] = f1_test
        best_model["Precision (Test)"] = prec_test
        best_model["Recall (Test)"] = rec_test
        best_model["Accuracy (Test)"] = acc_test

        metric["Seed"] = seed
        metric["Model"] = {'Name': best_model['Name'], 'Parameters': best_model['Parameters']}
        metric["F1-Score (Test)"] = f1_test
        metric["Precision (Test)"] = prec_test
        metric["Recall (Test)"] = rec_test
        metric["Accuracy (Test)"] = acc_test

        binary_metrics = metrics.binary_metrics(le.inverse_transform(y_test),
                                                le.inverse_transform(y_pred),
                                                class_map,
                                                return_results=True)

        for c in binary_metrics:
            for m in binary_metrics[c]:
                metric["class_" + c + "_" + m] = binary_metrics[c][m]


        results.append(metric)
        print("Done!\n")

    
    results_df = pd.DataFrame(results)
    return results_df

In [14]:
le = LabelEncoder()

featurized_data['y_act'] = le.fit_transform(featurized_data['y_act'])

featurized_data['GPT_5_4_detect_all_at_once'] = le.transform(featurized_data['GPT_5_4_detect_all_at_once'])
featurized_data['GPT_5_4_detect_all_at_once_fewshot'] = le.transform(featurized_data['GPT_5_4_detect_all_at_once_fewshot'])
featurized_data['Qwen_3_5_detect_all_at_once'] = le.transform(featurized_data['Qwen_3_5_detect_all_at_once'])
featurized_data['Qwen_3_5_detect_all_at_once_fewshot'] = le.transform(featurized_data['Qwen_3_5_detect_all_at_once_fewshot'])
featurized_data['Gemma_4_detect_all_at_once'] = le.transform(featurized_data['Gemma_4_detect_all_at_once'])
featurized_data['Gemma_4_detect_all_at_once_fewshot'] = le.transform(featurized_data['Gemma_4_detect_all_at_once_fewshot'])

In [15]:
feature_sets = {
    "Core Features + all-MiniLM-L6-v2 embeddings + GPT-5.4 Binary Features (Few-shot)": {
        'features': base_features_with_regex + gpt_5_4_fewshot_features,
        'embedding': 'all-minilm-l6-v2',
        'model': 'XGB',
    }
}

In [16]:
results = resample_and_evaluate(n_splits=100, feature_sets=feature_sets, featurized_data=featurized_data, raw_data=df_all)

	Resample 1 ... Done!

	Resample 2 ... Done!

	Resample 3 ... Done!

	Resample 4 ... Done!

	Resample 5 ... Done!

	Resample 6 ... Done!

	Resample 7 ... Done!

	Resample 8 ... Done!

	Resample 9 ... Done!

	Resample 10 ... Done!

	Resample 11 ... Done!

	Resample 12 ... Done!

	Resample 13 ... Done!

	Resample 14 ... Done!

	Resample 15 ... Done!

	Resample 16 ... Done!

	Resample 17 ... Done!

	Resample 18 ... Done!

	Resample 19 ... Done!

	Resample 20 ... Done!

	Resample 21 ... Done!

	Resample 22 ... Done!

	Resample 23 ... Done!

	Resample 24 ... Done!

	Resample 25 ... Done!

	Resample 26 ... Done!

	Resample 27 ... Done!

	Resample 28 ... Done!

	Resample 29 ... Done!

	Resample 30 ... Done!

	Resample 31 ... Done!

	Resample 32 ... Done!

	Resample 33 ... Done!

	Resample 34 ... Done!

	Resample 35 ... Done!

	Resample 36 ... Done!

	Resample 37 ... Done!

	Resample 38 ... Done!

	Resample 39 ... Done!

	Resample 40 ... Done!

	Resample 41 ... Done!

	Resample 42 ... Done!

	

In [17]:
results.to_csv('results/OurBestMethod.csv', index=False)

In [18]:
metrics = ["F1-Score (Dev)", "Precision (Dev)", "Recall (Dev)", "Accuracy (Dev)",
     "F1-Score (Test)", "Precision (Test)", "Recall (Test)", "Accuracy (Test)"]

for i, m in enumerate(metrics):
    if i == 4:
        print('\n')
    print(f'{m:<20}: {results[m].mean():>8.3f}')

F1-Score (Dev)      :    0.929
Precision (Dev)     :    0.935
Recall (Dev)        :    0.928
Accuracy (Dev)      :    0.938


F1-Score (Test)     :    0.924
Precision (Test)    :    0.928
Recall (Test)       :    0.924
Accuracy (Test)     :    0.936


In [19]:
stats = [
    'F1-Score (Dev)',
    'Precision (Dev)',
    'Recall (Dev)',
    'Accuracy (Dev)',
    'F1-Score (Test)',
    'Precision (Test)',
    'Recall (Test)',
    'Accuracy (Test)',
    'class_numeric_F1-Score',
    'class_numeric_Precision',
    'class_numeric_Recall',
    'class_numeric_Accuracy',
    'class_categorical_F1-Score',
    'class_categorical_Precision',
    'class_categorical_Recall',
    'class_categorical_Accuracy',
    'class_sentence_F1-Score',
    'class_sentence_Precision',
    'class_sentence_Recall',
    'class_sentence_Accuracy',
    'class_url_F1-Score',
    'class_url_Precision',
    'class_url_Recall',
    'class_url_Accuracy',
    'class_list_F1-Score',
    'class_list_Precision',
    'class_list_Recall',
    'class_list_Accuracy',
    'class_embedded-number_F1-Score',
    'class_embedded-number_Precision',
    'class_embedded-number_Recall',
    'class_embedded-number_Accuracy',
    'class_datetime_F1-Score',
    'class_datetime_Precision',
    'class_datetime_Recall',
    'class_datetime_Accuracy',
    'class_not-generalizable_F1-Score',
    'class_not-generalizable_Precision',
    'class_not-generalizable_Recall',
    'class_not-generalizable_Accuracy',
    'class_context-specific_F1-Score',
    'class_context-specific_Precision',
    'class_context-specific_Recall',
    'class_context-specific_Accuracy'
]

In [20]:
mean_class_specific = results[stats[8:]].mean().round(3)

In [21]:
for c in list(class_map.values()):
    print(f"### Class {c} ###\n")
    class_metrics = [f for f in list(mean_class_specific.keys()) if c in str(f).lower()]
    for m in class_metrics:
        print(f'{m.replace("class_", "").replace(c+"_", "", )}: {mean_class_specific[m]}')
    print('\n')
print('///////////////////////////////////////////////////////////////\n')

### Class numeric ###

F1-Score: 0.957
Precision: 0.949
Recall: 0.966
Accuracy: 0.978


### Class categorical ###

F1-Score: 0.934
Precision: 0.914
Recall: 0.957
Accuracy: 0.966


### Class datetime ###

F1-Score: 0.996
Precision: 0.996
Recall: 0.996
Accuracy: 0.999


### Class sentence ###

F1-Score: 0.939
Precision: 0.933
Recall: 0.946
Accuracy: 0.994


### Class url ###

F1-Score: 0.999
Precision: 0.999
Recall: 1.0
Accuracy: 1.0


### Class embedded-number ###

F1-Score: 0.979
Precision: 0.973
Recall: 0.985
Accuracy: 0.997


### Class list ###

F1-Score: 0.906
Precision: 0.897
Recall: 0.926
Accuracy: 0.996


### Class not-generalizable ###

F1-Score: 0.912
Precision: 0.935
Recall: 0.897
Accuracy: 0.982


### Class context-specific ###

F1-Score: 0.691
Precision: 0.753
Recall: 0.645
Accuracy: 0.96


///////////////////////////////////////////////////////////////

